# 01 â€” Pengumpulan Data: Kualitas Udara Indonesia

**Proyek portofolio Data Analyst** â€” analisis kualitas udara Indonesia (nasional per provinsi + studi kasus kota besar).

## Tujuan notebook
Mengunduh dan mengekstrak seluruh data mentah dari **sumber resmi pemerintah**, lalu menyimpannya ke `data/raw/` secara **reproducible**. Semua angka yang dipakai di tahap berikutnya harus dapat dilacak ke notebook ini.

## Sumber data (semua resmi)
| # | Dataset | Sumber |
|---|---------|--------|
| 1 | IKU Provinsi 2020â€“2024 (Tabel 2.10) | KLHK â€” *Statistik Lingkungan Hidup Vol. I 2025* (PDF) |
| 2 | IKLH Provinsi 2020â€“2024 (Tabel 2.1) | KLHK â€” publikasi yang sama |
| 3 | IKU Kabupaten/Kota 2020â€“2024 (Tabel 2.11) | KLHK â€” publikasi yang sama |
| 4 | Komponen Penyusun IKLH Provinsi 2024 | BPS (statistics-table) |
| 5 | Kepadatan Penduduk Provinsi 2021 | BPS (statistics-table) |

## Catatan metode & etika data
- **Tidak ada data yang dikarang.** Setiap file diunduh/diekstrak dari URL resmi yang tercatat di `DATA_SOURCES.md`.
- **IKU â‰  ISPU.** IKU (Indeks Kualitas Udara) skala 0â€“100, makin tinggi makin baik; dihitung dari NOâ‚‚ & SOâ‚‚. ISPU skala 0â€“500+, makin tinggi makin buruk.
- Data pemerintah dipakai dengan atribusi ke KLHK/BPS.


In [1]:
from pathlib import Path
import re, json, io
import requests
import pandas as pd
import pypdf

# --- lokasi proyek ---
ROOT = Path.cwd()
if (ROOT / "notebooks").exists():
    pass
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"}

KLHK_PDF_URL = ("https://statistik.kemenlh.go.id/admin-office/storage/publikasi/"
                "statistik-lingkungan-hidup-volume-i-tahun-2025.pdf")
KLHK_PDF = RAW / "KLHK_Statistik_Lingkungan_Hidup_Vol1_2025.pdf"

print("Proyek :", ROOT)
print("Data   :", RAW)
print("Sudah ada PDF KLHK:", KLHK_PDF.exists(), KLHK_PDF.stat().st_size if KLHK_PDF.exists() else "-")

Proyek : C:\Voicegame\analisis-kualitas-udara-indonesia
Data   : C:\Voicegame\analisis-kualitas-udara-indonesia\data\raw
Sudah ada PDF KLHK: True 15454455


## 1. Publikasi resmi KLHK (sumber utama tren 2020â€“2024)

Unduh PDF jika belum ada, lalu ekstrak tabel-tabelnya. Ketergantungan pada PDF dihindari di tahap analisis: kita ubah sekali menjadi CSV bersih.

In [2]:
def ensure_klhk_pdf():
    if KLHK_PDF.exists() and KLHK_PDF.stat().st_size > 1_000_000:
        print("PDF sudah tersedia, lewati unduh.")
        return
    print("Mengunduh PDF KLHK ...")
    r = requests.get(KLHK_PDF_URL, headers=HEADERS, timeout=180)
    r.raise_for_status()
    KLHK_PDF.write_bytes(r.content)
    print("Tersimpan:", KLHK_PDF, f"({KLHK_PDF.stat().st_size/1e6:.1f} MB)")

ensure_klhk_pdf()

PDF sudah tersedia, lewati unduh.


### 1a. Parser tabel *per provinsi* (IKU & IKLH)

Tabel pada PDF berformat teks polos; nama provinsi panjang ter-*wrap* ke beberapa baris. Parser di bawah menyatukan kembali nama yang terpotong lalu menangkap 5 nilai tahunan.

In [3]:
NUM = re.compile(r"^\d+(?:\.\d+)?$")
DASHES = {"-", "\u2013", "\u2014", "\ufffd", "\u25a0", "\xad", "â€“"}
SKIP = ["Pengendalian Pencemaran", "Statistik Kementerian", "Tabel 2", "Sumber",
        "No Provinsi", "Indeks Kualitas", "|", "Kabupaten/Kota", "Ekoregion"]
YEARLINE = re.compile(r"^(20\d\d\s*)+$")
COLLINE = re.compile(r"^\d(\s+\d)+$")

PROV_NORM = {
    "Nanggroe Aceh Darussalam": "ACEH",
    "DI Yogyakarta": "DI YOGYAKARTA",
    "Daerah Istimewa Yogyakarta": "DI YOGYAKARTA",
    "Kepulauan Bangka Belitung": "KEP. BANGKA BELITUNG",
    "Kepulauan Riau": "KEP. RIAU",
    "Nasional": "INDONESIA",
}

def _is_val(t):
    return bool(NUM.match(t)) or t in DASHES

def parse_klhk_province_pages(reader, page_idxs):
    rows, buffer = [], []
    for pi in page_idxs:
        for line in (reader.pages[pi].extract_text() or "").splitlines():
            l = line.strip()
            if not l or any(s in l for s in SKIP) or YEARLINE.match(l) or COLLINE.match(l):
                continue
            toks = l.split()
            vals, i = [], len(toks)
            while i > 0 and _is_val(toks[i - 1]):
                vals.insert(0, toks[i - 1]); i -= 1
            if len(vals) == 5:
                name = " ".join(buffer + [t for t in toks[:i] if not NUM.match(t)]).strip()
                buffer = []
                rows.append((name, vals))
            else:
                buffer += [t for t in toks if not NUM.match(t)]
    out = []
    for name, vals in rows:
        rec = {"provinsi": PROV_NORM.get(name, name.upper()), "provinsi_raw": name}
        for yr, v in zip([2020, 2021, 2022, 2023, 2024], vals):
            rec[yr] = None if v in DASHES else float(v)
        out.append(rec)
    return pd.DataFrame(out)[["provinsi", "provinsi_raw", 2020, 2021, 2022, 2023, 2024]]

reader = pypdf.PdfReader(KLHK_PDF)

iku_prov = parse_klhk_province_pages(reader, [144, 145])     # Tabel 2.10
iklh_prov = parse_klhk_province_pages(reader, [83, 84])      # Tabel 2.1

iku_prov.to_csv(RAW / "klhk_iku_provinsi_2020_2024.csv", index=False, encoding="utf-8-sig")
iklh_prov.to_csv(RAW / "klhk_iklh_provinsi_2020_2024.csv", index=False, encoding="utf-8-sig")
print("IKU provinsi :", iku_prov.shape, "| IKLH provinsi :", iklh_prov.shape)
iku_prov.head(8)

IKU provinsi : (39, 7) | IKLH provinsi : (39, 7)


,provinsi,provinsi_raw,2020,2021,2022,2023,2024
0,ACEH,Aceh,89.5,89.6,90.6,90.9,92.81
1,SUMATERA UTARA,Sumatera Utara,89.2,89.6,89.7,90.9,92.26
2,SUMATERA BARAT,Sumatera Barat,90.4,90.2,90.7,90.5,92.27
3,RIAU,Riau,90.4,90.1,90.7,90.9,89.66
4,JAMBI,Jambi,85.7,87.1,89.9,90.6,92.98
5,SUMATERA SELATAN,Sumatera Selatan,86.6,86.3,89.1,87.8,90.25
6,BENGKULU,Bengkulu,90.5,90.8,91.3,92.5,94.13
7,LAMPUNG,Lampung,85.5,85.5,87.3,88.0,90.94


### 1b. Parser tabel *kabupaten/kota* (Tabel 2.11)

Tabel ini memuat 514 baris. Kolom terakhir (`ekoregion`) berada setelah 5 nilai. Sekitar 97% baris berhasil diekstrak; sisanya gagal karena nama terpotong antar-halaman.

In [4]:
CITY_SKIP = ["Pengendalian Pencemaran", "Statistik Kementerian", "Tabel 2", "Sumber",
             "No Provinsi", "Indeks Kualitas", "|", "Ekoregion"]

def parse_klhk_city_pages(reader, page_idxs):
    rows, buffer = [], []
    for pi in page_idxs:
        for line in (reader.pages[pi].extract_text() or "").splitlines():
            l = line.strip()
            if not l or any(s in l for s in CITY_SKIP) or YEARLINE.match(l) or COLLINE.match(l):
                continue
            toks = l.split()
            if (len(toks) >= 6 and all(_is_val(t) for t in toks[-6:-1])
                    and any(NUM.match(t) for t in toks[-6:-1]) and not _is_val(toks[-1])):
                cand, ekoregion, name_toks = toks[-6:-1], toks[-1], toks[:-6]
            elif len(toks) >= 5 and all(_is_val(t) for t in toks[-5:]):
                cand, ekoregion, name_toks = toks[-5:], None, toks[:-5]
            else:
                buffer += [t for t in toks if not NUM.match(t)]
                continue
            name = " ".join(buffer + [t for t in name_toks if not NUM.match(t)]).strip()
            buffer = []
            rows.append((name, cand, ekoregion))
    data = []
    for name, vals, eko in rows:
        rec = {"nama": name, "ekoregion": eko}
        for yr, v in zip([2020, 2021, 2022, 2023, 2024], vals):
            rec[yr] = None if v in DASHES else float(v)
        data.append(rec)
    return pd.DataFrame(data)[["nama", 2020, 2021, 2022, 2023, 2024, "ekoregion"]]

iku_city = parse_klhk_city_pages(reader, range(146, 185))
iku_city.to_csv(RAW / "klhk_iku_kabkota_2020_2024.csv", index=False, encoding="utf-8-sig")
print("IKU kab/kota:", iku_city.shape)
print("Kota besar terverifikasi ada:",
      iku_city[iku_city["nama"].str.contains("Kota Surabaya|Kota Bandung|Kota Medan|Kota Jakarta Pusat")]["nama"].tolist())

IKU kab/kota: (499, 7)
Kota besar terverifikasi ada: ['DKI Jakarta Kota Jakarta Pusat', 'Jawa Barat Kota Bandung', 'Jawa Timur Kota Surabaya', 'Sumatera Utara Kota Medan']


## 2. Tabel BPS (statistics-table)

Halaman BPS menyematkan payload JSON di HTML. Kita ekstrak array `vervar` (provinsi), `turvar` (komponen), `tahun`, dan `datacontent` (nilai) tanpa perlu API key.

In [5]:
def bps_get(url):
    return requests.get(url, headers=HEADERS, timeout=40).text

def _bps_arr(html, field):
    m = re.search(r'\\"' + field + r'\\":\[(.*?)\]', html, re.S)
    return json.loads("[" + m.group(1).replace('\\"', '"') + "]") if m else None

def _bps_dic(html, field):
    m = re.search(r'\\"' + field + r'\\":\{(.*?)\}', html, re.S)
    return json.loads("{" + m.group(1).replace('\\"', '"') + "}") if m else None

# --- Komponen Penyusun IKLH per provinsi (2024) ---
URL_IKLH = ("https://www.bps.go.id/id/statistics-table/2/MjUzMiMy/"
            "komponen-penyusun-indeks-kualitas-lingkungan-hidup-menurut-provinsi.html")
h = bps_get(URL_IKLH)
turvar = _bps_arr(h, "turvar")
vervar = _bps_arr(h, "vervar")
dc = _bps_dic(h, "datacontent")
tmap = {v["val"]: v["label"] for v in turvar}
pmap = {v["val"]: v["label"] for v in vervar}
rec = {}
for key, val in dc.items():
    rec.setdefault(pmap[int(key[:4])], {})[tmap[int(key[8:12])]] = val
komp = pd.DataFrame.from_dict(rec, orient="index").reset_index().rename(columns={"index": "provinsi"})
komp = komp[["provinsi"] + [v["label"] for v in turvar]]
komp.to_csv(RAW / "bps_komponen_iklh_provinsi_2024.csv", index=False, encoding="utf-8-sig")
print("BPS komponen IKLH 2024:", komp.shape)
komp.head(4)

BPS komponen IKLH 2024: (39, 7)


,provinsi,Indeks Kualitas Udara,Indeks Kualitas Air,Indeks Kualitas Tutupan Lahan (IKTL),Indeks Kualitas Lahan (IKL),Indeks Kualitas Ekosistem Gambut (IKEG),Indeks Kualitas Air Laut (IKAL)
0,LAMPUNG,90.94,62.97,38.48,38.49,43.28,77.35
1,JAWA BARAT,83.68,46.86,42.07,42.07,NaN,88.34
2,PAPUA PEGUNUNGAN,93.81,58.89,99.42,99.42,97.70,NaN
3,PAPUA BARAT DAYA,95.79,60.00,100.00,100.00,97.96,84.56


In [6]:
# --- Kepadatan penduduk per provinsi (2021) ---
URL_DENS = ("https://www.bps.go.id/id/statistics-table/2/MTQxIzI=/"
            "kepadatan-penduduk-menurut-provinsi.html")
h = bps_get(URL_DENS)
vervar = _bps_arr(h, "vervar")
thn = _bps_arr(h, "tahun")[0]["label"]
dc = _bps_dic(h, "datacontent")
pmap = {v["val"]: v["label"] for v in vervar}
dens = pd.DataFrame([{"provinsi": pmap[int(k[:4])], "tahun": thn, "kepadatan_penduduk": v}
                     for k, v in dc.items()]).sort_values("provinsi").reset_index(drop=True)
dens.to_csv(RAW / "bps_kepadatan_penduduk_provinsi_2021.csv", index=False, encoding="utf-8-sig")
print("BPS kepadatan penduduk:", dens.shape, "| tahun:", thn)
dens.head(4)

BPS kepadatan penduduk: (35, 3) | tahun: 2021


,provinsi,tahun,kepadatan_penduduk
0,ACEH,2021,92
1,BALI,2021,755
2,BANTEN,2021,1248
3,BENGKULU,2021,102


## 3. Verifikasi silang & inventaris

Bandingkan IKU 2024 dari KLHK dengan BPS, lalu tampilkan daftar file di `data/raw/`.

In [7]:
klhk24 = iku_prov.set_index("provinsi")[2024]
bps24 = komp.set_index("provinsi")["Indeks Kualitas Udara"]
cek = pd.DataFrame({"IKU_2024_KLHK": klhk24, "IKU_2024_BPS": bps24}).dropna()
cek["selisih"] = (cek["IKU_2024_KLHK"] - cek["IKU_2024_BPS"]).abs()
print("Jumlah provinsi terbandingkan:", len(cek))
print("Selisih maksimum:", round(cek["selisih"].max(), 2))
print(cek.head(6))

print("\n--- Inventaris data/raw ---")
for f in sorted(RAW.glob("*")):
    print(f"  {f.name:45s} {f.stat().st_size/1024:8.1f} KB")

Jumlah provinsi terbandingkan: 39
Selisih maksimum: 0.03
               IKU_2024_KLHK  IKU_2024_BPS  selisih
provinsi                                           
ACEH                   92.81         92.81      0.0
BALI                   92.12         92.12      0.0
BANTEN                 69.84         69.84      0.0
BENGKULU               94.13         94.13      0.0
DI YOGYAKARTA          90.44         90.44      0.0
DKI JAKARTA            72.59         72.59      0.0

--- Inventaris data/raw ---
  bps_kepadatan_penduduk_provinsi_2021.csv           0.8 KB
  bps_komponen_iklh_provinsi_2024.csv                2.0 KB
  klhk_iklh_provinsi_2020_2024.csv                   2.2 KB
  klhk_iku_kabkota_2020_2024.csv                    36.1 KB
  klhk_iku_provinsi_2020_2024.csv                    2.0 KB
  KLHK_Statistik_Lingkungan_Hidup_Vol1_2025.pdf  15092.2 KB


## Keterbatasan pengumpulan data
- **Kendaraan bermotor per provinsi**: tabel BPS bertipe dinamis (data tidak disematkan di HTML; di-*load* via `web-api.bps.go.id` yang butuh kunci). **Dikecualikan** dari analisis atas keputusan pemilik proyek.
- **Kepadatan penduduk**: hanya tahun 2021 tersedia pada endpoint tipe-2.
- **IKU kab/kota**: 499 dari 514 baris terekstrak; seluruh kota besar target terverifikasi ada.
- **Data harian (ISPU)**: tidak digunakan; analisis memakai IKU tahunan resmi yang sebanding antarwilayah.
